# DAST Retrieval Engine — Run Notebook

This notebook replaces the old top-to-bottom `run.py`. Each step is isolated so you
can run just the part you need instead of triggering the whole pipeline every time.

**How to use it:**

| Step | What it does | When to run | Needs |
|------|--------------|-------------|-------|
| 1 | Parse `document2.pdf` → DAST JSON | Once per document | fast, no LLM |
| 2 | Browse the tree to author `questions.jsonl` | Only while writing questions | **interactive** |
| 3.1 | Engine-only retrieval sanity check | Every time you tweak the engine | fast, no LLM |
| 4 | Full DAST vs RAG bake-off | Occasionally | torch/faiss + HF token |
| Summary | Pretty-print benchmark results | After Step 4 | — |

> Step 2 is an **authoring helper** — it waits for your keyboard input. That's why it
> lives in its own cell: you simply don't run it during a normal pipeline pass.

## Step 1 — Parse `document2.pdf` → DAST

Writes `benchmark/document2.dast.json`. Run once per document (or after changing the parser).

In [ ]:
# PDFParser is our OWN class (parsers/pdf_parser.py) - the only thing to install
# is pdfplumber. This kernel is a uv venv (no pip module), so we shell out to
# `uv pip install`, targeting THIS kernel's python via sys.executable.
import sys
!uv pip install pdfplumber --python {sys.executable}


In [3]:
import json
from parsers.pdf_parser import PDFParser

r = PDFParser().parse('benchmark/document2.pdf')
json.dump(r.to_dict(), open('benchmark/document2.dast.json', 'w'), indent=2)
print('wrote benchmark/document2.dast.json')

wrote benchmark/document2.dast.json


## Step 2 — Browse the tree to author `questions.jsonl`  *(interactive)*

Search the parsed doc for text; it prints the matching `node_id`s (which you paste into
`questions.jsonl` as `ground_truth_node_ids`).

**Run this cell only when you're authoring questions** — it waits for your input.
Verify each `ground_truth_node_id` against this output before trusting the metrics.

In [4]:
import json

d = json.load(open('benchmark/document2.dast.json'))

def walk(n):
    yield n
    for c in n['children']:
        yield from walk(c)

term = input('search term: ').lower()
for n in walk(d):
    txt = ((n.get('title') or '') + ' ' + (n.get('text') or '')).lower()
    if term in txt:
        loc = n.get('physical_location') or {}
        label = (n.get('title') or n.get('text') or '')[:80].replace(chr(10), ' ')
        print(n['node_id'], '| p', loc.get('page_start'), '|', label)

# e.g. type: 'job title' -> gives doc.10.11.6 (p7) etc. to use as ground_truth_node_ids

doc | p None | document2.pdf
doc.1 | p 1 | 1
doc.2 | p 2 | Copyright © 2014 Nikos Moraitakis
doc.3 | p 2 | All rights reserved
doc.4 | p 2 | It’s all yours. You can help yourself to any of the job descriptions in our comp
doc.5 | p 2 | available in PDF form or if you prefer there’s a link to a downloadable Word doc
doc.6 | p 2 | each section. Enjoy, customise and never face a blank page again.
doc.7 | p 2 | Publisher: Workable
doc.8 | p 2 | Editor: Daniel Howden
doc.9 | p 2 | Author: Eleni Kourmentza
doc.10 | p 2 | Designers: Danae Panopoulou, Panagiotis Efthymiou
doc.11 | p 2 | This book is alive. Go to:
doc.12 | p 2 | http://get.workable.com/job-description-compendium-download
doc.13 | p 2 | to find the latest version.
doc.14 | p 2 | Follow @Workable on twitter
doc.15 | p 2 | 2
doc.16 | p 3 | 3
doc.17 | p 4 | CONTENTS
doc.17.1 | p 4 | Foreword by Kirsti Grant 5
doc.17.2 | p 4 | The Style Guide For Job Descriptions 7
doc.17.3 | p 4 | Pimp Your Job Descriptions Examples 10
doc.17.4 | p

## Step 3.1 — Engine-only sanity check  *(no LLM, instant)*

Runs the retriever over every question and reports how often the ground-truth node is
returned. Two views:

* **Strict** — the exact ground-truth `node_id` is in the top-k.
* **Ancestor/descendant** — a related node (parent/child on the same path) is in the top-k,
  which is a legitimate hit in hierarchical retrieval.

In [5]:
import json
from engine.index import DASTIndex
from engine.retriever import DASTRetriever

def related(a, b):
    return a == b or b.startswith(a + '.') or a.startswith(b + '.')

idx = DASTIndex.from_json('benchmark/document2.dast.json')
eng = DASTRetriever(idx, strategy='best_first')

strict = lenient = total = 0
for line in open('benchmark/questions.jsonl'):
    line = line.strip()
    if not line:
        continue
    ex = json.loads(line)
    total += 1
    got = [x.node_id for x in eng.retrieve(ex['question'], top_k=8)]
    gt = ex['ground_truth_node_ids'][0]
    s = gt in got
    l = any(related(n, gt) for n in got)
    strict += s
    lenient += l
    print(('L-HIT' if l else 'MISS '), 'S' if s else '.', ex['qid'], '| gt', gt, '| top5', got[:5])

print(f'\nSTRICT hit@8: {strict}/{total}  |  ANCESTOR/DESCENDANT hit@8: {lenient}/{total}')

MISS  . q001 | gt doc.7 | top5 ['doc.19.12.1.5', 'doc.19.12.1.10', 'doc.19.12.1.17', 'doc.19', 'doc.19.12']
L-HIT . q002 | gt doc.19.1 | top5 ['doc.19', 'doc.19.12.1.12', 'doc.19.12.2.3', 'doc.19.12.2.4', 'doc.19.12.2.5']
MISS  . q003 | gt doc.19.11.13.8.10 | top5 ['doc.19.12.2.4', 'doc.19.12.1.5', 'doc.19.12.1.12', 'doc.19.12.2.3', 'doc.19.12.2.5']
L-HIT S q004 | gt doc.19.11.13.6 | top5 ['doc.19.11.9', 'doc.19.12.2.4', 'doc.19.13.1.6', 'doc.18.9.9', 'doc.19.11.13.6']
MISS  . q005 | gt doc.19.11.6.7 | top5 ['doc.14', 'doc.19.32.4.10', 'doc.20.3.3.4', 'doc.21.4.4.8', 'doc.21.9.5.6']
L-HIT . q006 | gt doc.18.9.8 | top5 ['doc.18.9', 'doc.18', 'doc.22.4', 'doc.22.5', 'doc.22.4.3']
L-HIT . q007 | gt doc.19.12.2.4.1 | top5 ['doc.19.12.2.3', 'doc.19.12.2.4', 'doc.18.9.15', 'doc.19.12.2.5', 'doc.19.12.1.4']
L-HIT . q008 | gt doc.19.4 | top5 ['doc.19', 'doc.19.12.1.12', 'doc.19.12.2.3', 'doc.19.12.2.4', 'doc.19.12.2.5']
L-HIT . q009 | gt doc.19.11.6 | top5 ['doc.19.12.2.4', 'doc.19', 'doc.19.1

### (Optional) Spot-check a single query

Handy for eyeballing scores and matched terms on one question.

In [6]:
for x in eng.retrieve('tips for writing job descriptions', top_k=3):
    print(round(x.score, 3), x.node_id, x.matched_terms)

0.6 doc.19 ['job', 'description']
0.6 doc.19.12 ['job', 'description']
0.6 doc.19.13 ['job', 'description']


## Step 4 - Full bake-off: DAST vs RAG baseline  *(heavy)*

Install the heavy deps + OCR engine, then drop your HuggingFace token in `.env`:

```bash
uv pip install -r RAG/requirements.txt
brew install tesseract   # optional - only for OCR of image-only PDF pages
```

Put your token in the repo-root **`.env`** file (copy `.env.example`):

```
HF_TOKEN=hf_your_token_here
```

Your HF account must have accepted the Llama 3.2 license. The next cell loads
that token and pins the models. **First run downloads the weights (~6 GB) into
`~/.cache/huggingface`; later runs reuse them.**

> Networking / troubleshooting notes live in **`RUN_GUIDE.md`**.


In [ ]:
import os

# Load HF_TOKEN from the repo-root .env file (copy .env.example -> .env).
# python-dotenv ships with RAG/requirements.txt; a shell `export HF_TOKEN=` also works.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Models used for BOTH the DAST answer generator and the RAG baseline.
os.environ["DEFAULT_ANSWER_MODEL"]   = "meta-llama/Llama-3.2-3B-Instruct"  # DAST generator
os.environ["DEFAULT_BASELINE_MODEL"] = "meta-llama/Llama-3.2-3B-Instruct"  # RAG baseline

assert os.getenv("HF_TOKEN"), (
    "No HF_TOKEN found. Add it to the .env file at the repo root "
    "(HF_TOKEN=hf_xxx) or export it in your shell before running this cell."
)

print("answer  model:", os.environ["DEFAULT_ANSWER_MODEL"])
print("baseline model:", os.environ["DEFAULT_BASELINE_MODEL"])


In [8]:
from eval.run_benchmark import run_benchmark

result = run_benchmark(
    dast_path='benchmark/document2.dast.json',
    questions_path='benchmark/questions.jsonl',
    pdf_path='benchmark/document2.pdf',
    output_dir='results',
    top_k=8,
)
print('Benchmark complete:', result)

/Users/m0s182j/Library/CloudStorage/OneDrive-WalmartInc/Desktop/Repos/llm-over-fdf/src/text_to_sql/sql_to_text/bim-file/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: Unable to import RAG baseline dependencies

## Summary — pretty-print the benchmark results

In [ ]:
import json

data = json.load(open(result['output_path']))
print('=== SUMMARY ===')
print(json.dumps(data['summary'], indent=2))